## Multi-Agent Research Team with CrewAI and Gemini

This project builds a small team of AI agents that work together to produce a short report on a topic. Each agent is powered by a large language model (Google Gemini) and has its own role: a Researcher collects the key facts, a Writer turns them into a short report, and an Editor checks and improves the report. We build the team with the CrewAI library. The objective is in order to show how several specialized agents can share a task, and compare the team's result with the result of a single prompt to the same model.

## Approach
1. Set up CrewAI and connect it to the Gemini model with an API key
2. Define three agents (Researcher, Writer, Editor) with a role, a goal, and a short backstory
3. Define three tasks, where the output of one task is the input of the next
4. Assemble the crew and run it sequentially on a topic
5. Read the result and the output of each agent
6. Compare the crew with a single prompt (quality, number of calls, tokens, and time)
7. Discuss the limitations: the agents do not search the web, so they can make mistakes
8. Available at Streamlit

In [8]:
# set up the crewAI
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 14.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
bigframes 2.48.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [9]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key and press Enter: ")
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"

key = os.environ["GEMINI_API_KEY"]
print(key[:4], len(key))

Paste your Gemini API key and press Enter: ··········
AQ.A 53


In [10]:
from crewai import LLM
llm = LLM(model="gemini/gemini-3.8-flash", temperature=0.7)
print(llm.call("Say hello in one short sentence."))

Hello, I hope you are having a wonderful day!


In [15]:
import os, time, requests
from typing import Any
from crewai import BaseLLM


class GeminiRestLLM(BaseLLM):
    """Gemini via direct REST calls: key in the x-goog-api-key header, automatic retries, and fallback models."""

    fallback_models: list[str] = []

    def _post(self, model, body):
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
        headers = {"x-goog-api-key": os.environ["GEMINI_API_KEY"], "Content-Type": "application/json"}
        r = None
        for attempt in range(4):                                   # busy or rate-limited: wait and retry
            r = requests.post(url, headers=headers, json=body, timeout=120)
            if r.status_code not in (429, 500, 502, 503, 504):
                break
            time.sleep(2 ** (attempt + 1))                         # 2, 4, 8, 16 seconds
        return r

    def call(self, messages, tools=None, callbacks=None, available_functions=None,
             from_task=None, from_agent=None, response_model=None, **kwargs) -> Any:
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]
        system = "\n".join(m["content"] for m in messages if m["role"] == "system")
        contents = [
            {"role": "model" if m["role"] == "assistant" else "user", "parts": [{"text": m["content"]}]}
            for m in messages if m["role"] != "system"
        ]
        body = {"contents": contents, "generationConfig": {"temperature": self.temperature if self.temperature is not None else 0.7}}
        if system:
            body["systemInstruction"] = {"parts": [{"text": system}]}
        if self.stop:
            body["generationConfig"]["stopSequences"] = list(self.stop)[:5]

        for model in [self.model] + list(self.fallback_models):    # try the main model first, then the fallbacks
            r = self._post(model, body)
            if r.status_code == 200:
                parts = r.json()["candidates"][0]["content"]["parts"]
                return "".join(p.get("text", "") for p in parts)
            if r.status_code not in (429, 500, 502, 503, 504):
                break
        raise RuntimeError(f"Gemini API error {r.status_code}: {r.text[:300]}")


llm = GeminiRestLLM(model="gemini-3.8-flash", fallback_models=["gemini-3.5-flash-lite"], temperature=0.7)
print(llm.call("Say hello in one short sentence."))

Hello, it is wonderful to meet you!


In [17]:
import time
from crewai import Agent, Task, Crew, Process

print("LLM class in use:", type(llm).__name__)          # GeminiRestLLM görünmeli

researcher = Agent(
    role="Researcher",
    goal="Collect the key facts about {topic}",
    backstory="You are a careful analyst. You only state facts you are confident about, "
              "you mark uncertain points with '(uncertain)', and you never invent statistics or sources.",
    llm=llm, allow_delegation=False, verbose=False,
)
writer = Agent(
    role="Writer",
    goal="Write a clear and short report about {topic} using only the researcher's notes",
    backstory="You are a clear technical writer who explains things simply and does not add facts that are not in the notes.",
    llm=llm, allow_delegation=False, verbose=False,
)
editor = Agent(
    role="Editor",
    goal="Fact-check the report and return the final version",
    backstory="You are a strict editor. You remove or soften claims that are not supported, and you keep the report short and well structured.",
    llm=llm, allow_delegation=False, verbose=False,
)

research_task = Task(
    description="Collect 6 key facts about {topic}. Mark any uncertain fact with '(uncertain)'. Do not invent numbers or sources.",
    expected_output="A bullet list of 6 short facts.",
    agent=researcher,
)
writing_task = Task(
    description="Write a report of about 150 words about {topic}, using only the facts in the research notes.",
    expected_output="A short report in clear English.",
    agent=writer, context=[research_task],
)
editing_task = Task(
    description="Review the report against the research notes. Remove unsupported claims, fix unclear sentences, "
                "and return the final report in markdown with a title and a short 'Limitations' line.",
    expected_output="The final report in markdown.",
    agent=editor, context=[writing_task, research_task],
)

crew = Crew(agents=[researcher, writer, editor], tasks=[research_task, writing_task, editing_task],
            process=Process.sequential, verbose=False)

start = time.time()
try:
    result = await crew.kickoff_async(inputs={"topic": "solar energy"})
    print(f"Finished in {time.time() - start:.0f} seconds\n")
    print(result.raw)
except Exception as e:
    print(type(e).__name__, ":", str(e)[:300])

LLM class in use: GeminiRestLLM
Finished in 19 seconds

# Solar Energy Overview

Solar energy is a renewable resource derived from the radiant light and heat produced by nuclear fusion within the sun.

Electricity is generated from this resource through direct or indirect methods:

* **Photovoltaic (PV) Systems:** PV cells convert sunlight directly into electricity via the photovoltaic effect, predominantly using silicon as the semiconductor material.
* **Concentrated Solar Power (CSP):** CSP systems generate electricity indirectly by using mirrors or lenses to concentrate sunlight to heat a fluid, which drives a conventional steam turbine.

Operationally, generating electricity from solar panels produces zero direct greenhouse gas emissions and consumes no water for fuel combustion.

**Limitations:** Solar power is inherently intermittent due to variations in daylight availability, weather conditions, and seasonal angles of incidence; while PV panels can still operate under overcast s

In [18]:
for name, task in [("RESEARCHER", research_task), ("WRITER", writing_task), ("EDITOR", editing_task)]:
    print("=" * 20, name, "=" * 20)
    print(task.output.raw, "\n")

==================== RESEARCHER ====================
Here are 6 key facts about solar energy:

* Solar energy is a renewable resource derived from the radiant light and heat produced by nuclear fusion within the sun.
* Photovoltaic (PV) cells convert sunlight directly into electricity via the photovoltaic effect, predominantly using silicon as the semiconductor material.
* Concentrated solar power (CSP) systems generate electricity indirectly by using mirrors or lenses to concentrate sunlight to heat a fluid, which drives a conventional steam turbine.
* Generating electricity from operational solar panels produces zero direct greenhouse gas emissions and consumes no water for fuel combustion.
* Solar power is intermittent because energy production depends on daylight availability, weather conditions, and seasonal angles of incidence.
* Solar photovoltaic panels can still generate electricity during overcast or cloudy days by utilizing diffuse light, although power output is significant

In [19]:
import re
import pandas as pd

single_prompt = ("Write a report of about 150 words about solar energy in clear English, with a title and a short "
                 "'Limitations' line. Do not invent statistics or sources.")
start = time.time()
single = llm.call(single_prompt)
single_seconds = time.time() - start
print(single)

**Report: The Role and Expansion of Solar Energy**

Solar energy is one of the most abundant and rapidly expanding sources of renewable power. It works by capturing radiant light and heat from the sun using technologies such as photovoltaic panels and solar thermal collectors. These systems convert sunlight into electricity or heat for residential, commercial, and industrial use.

A primary advantage of solar power is environmental sustainability. Generating electricity from the sun produces no direct greenhouse gas emissions or air pollutants during operation, making it a key tool in mitigating global climate change. Additionally, solar power is versatile; it can be deployed on small rooftops to power single households or across vast utility-scale solar farms to supply the wider electrical grid. 

**Limitations:** Solar energy production is inherently intermittent due to nighttime and cloudy weather, requiring reliable energy storage systems and high upfront installation costs.


In [20]:
crew_text = result.raw

def numbers_in(text):
    return len(re.findall(r"\d+(?:\.\d+)?", text))

pd.DataFrame({
    "Single prompt": {"LLM calls": 1, "words": len(single.split()), "numbers mentioned": numbers_in(single)},
    "Crew (3 agents)": {"LLM calls": 3, "words": len(crew_text.split()), "numbers mentioned": numbers_in(crew_text)},
})

,Single prompt,Crew (3 agents)
LLM calls,1,3
words,144,146
numbers mentioned,0,0


In [21]:
async def compare(topic):
    t0 = time.time()
    single = llm.call(f"Write a report of about 150 words about {topic} in clear English, with a title and a short "
                      "'Limitations' line. Do not invent statistics or sources.")
    single_seconds = time.time() - t0

    t0 = time.time()
    crew_result = await crew.kickoff_async(inputs={"topic": topic})
    crew_seconds = time.time() - t0

    table = pd.DataFrame({
        "Single prompt": {"LLM calls": 1, "seconds": round(single_seconds), "words": len(single.split()), "numbers mentioned": numbers_in(single)},
        "Crew (3 agents)": {"LLM calls": 3, "seconds": round(crew_seconds), "words": len(crew_result.raw.split()), "numbers mentioned": numbers_in(crew_result.raw)},
    })
    return single, crew_result.raw, table

single_text, crew_text, table = await compare("Togg, the Turkish electric car")
print("---- SINGLE PROMPT ----\n" + single_text + "\n\n---- CREW ----\n" + crew_text)
table

---- SINGLE PROMPT ----
**Togg: Turkey’s National Electric Vehicle Initiative**

Togg, officially known as Türkiye's Automobile Joint Venture Group, is Turkey's first indigenous electric vehicle brand. Established in 2018 by a consortium of prominent Turkish industrial and technological enterprises, the venture aims to transform the nation's manufacturing sector toward electrification and proprietary software.

Production began at the company’s dedicated facility in Gemlik, Bursa, in late 2022. Togg's debut model is the T10X, a fully electric C-segment SUV developed in collaboration with the Italian design house Pininfarina. The vehicle features smart connectivity, integrated digital services, and modern driver-assistance systems. While initial rollouts have supplied domestic consumers and state institutions, Togg plans to gradually expand distribution into select European markets. Overall, the project reflects Turkey’s ambition to establish domestic intellectual property in the global

,Single prompt,Crew (3 agents)
LLM calls,1,3
seconds,67,83
words,153,183
numbers mentioned,3,11
